## Environment setup

Configure PyTorch precision and suppress non-critical warnings before any model
imports. `float32_matmul_precision='high'` enables TF32 on Ampere-class GPUs,
giving a significant training speedup with negligible accuracy loss.

In [1]:
import torch
torch.set_float32_matmul_precision('high')
import warnings
warnings.filterwarnings("ignore")

<div style="display: flex; gap: 10px;">
  <img src="../images/HOOPS_AI.jpg" style="width: 20%;">

# Training Your Custom HOOPS Embeddings Model


The `EmbeddingFlowModel` is a specialized FlowModel implementation designed for training shape-embedding models from CAD data using self-supervised learning.

This enables data scientists to train custom HOOPS embeddings models on their own CAD datasets.

In [ ]:
import hoops_ai
import os
import sys

license_key = os.environ.get("HOOPS_AI_LICENSE")
if not license_key:
    sys.exit("HOOPS_AI_LICENSE environment variable is required.")

hoops_ai.set_license(license_key, validate=True)

------------------------------------------------------------
HOOPS AI
------------------------------------------------------------
  Platform      : Windows 11
  Architecture  : AMD64
  Python        : 3.12.10
------------------------------------------------------------
  Core          : hoops-ai             1.1.0  (build: ed23c844 2026-06-12T14:14:13Z)
  CAD Access    : hoops-exchange       26.2.0  (build: 1e11169 2026-06-12T10:38:16Z)
  Conversion    : hoops-converter      26.1.1  (build: 00dc9f6 2026-06-12T10:22:46Z)
  Insights      : hoops-web-viewer     26.1.1  (build: d30058f 2026-06-12T10:22:25Z)
------------------------------------------------------------
[OK] HOOPS AI License: Valid


## Dataset

This notebook uses the **TMCAD** dataset (Truly Mechanical CAD Dataset), introduced in:

> Zou, Q., & Zhu, L. (2025). Bringing attention to CAD: Boundary representation learning via transformer. *Computer-Aided Design*, 103940. Elsevier.


**Download links:**
- TMCAD v2 (2025-11-02, recommended): <https://pan.zju.edu.cn/share/218d10a88e8c18f5b96e94a7e0>
- Dataset documentation: <https://github.com/Qiang-Zou/BRT/blob/main/DATASET.md>

TMCAD v2 is a cleaned and verified collection of ~10,000 CAD models in STEP (`.stp`) format across 10 mechanical part categories (bearing, bolt-screw, bracket, flange, gear, nut, shaft, coupling, pulley, spring). Released under the **GPL-3.0 license**.

## Step 1: Prepare your CAD dataset

Training `EmbeddingFlowModel` requires a preprocessing pipeline that gathers CAD
files and encodes them into the tensor format the model expects. The pipeline runs
in parallel across workers, producing a `.dataset` store and an `.infoset` metadata
file that the `DatasetLoader` (Step 2) will consume.

In [ ]:
import pathlib
import sys

path_tmcad_v2_dataset = os.environ.get("PATH_DATASET_TMCAD")
if not path_tmcad_v2_dataset:
    sys.exit("PATH_DATASET_TMCAD environment variable is required.")

datasources_dir = pathlib.Path(path_tmcad_v2_dataset)

### Parallel encoding pipeline

The flow processes each CAD file through two tasks defined in
`scripts/cad_tasks_embeddings.py`:

1. **`gather_cad_files`** — walks `datasources_dir` and emits one record per file.
2. **`encode_data_for_ml_training`** — converts each file to the B-rep tensor
   representation the embedding model expects (face types, edge connectivity, etc.).

`max_workers` controls parallelism. Set it to the number of physical CPU cores
available (or slightly below to leave headroom). Results are written to `flows_outputdir`
as a `.dataset` store and an `.infoset` metadata file.

In [4]:
# We define our tasks in a separate file for multiprocessing compatibility.
from scripts.cad_tasks_embeddings import EmbeddingModel, flows_outputdir, get_flow_name, gather_cad_files, encode_data_for_ml_training

# Create and run the data flow
flow_name = get_flow_name() 

data_prep = hoops_ai.create_flow(
    name=flow_name,
    tasks=[gather_cad_files, encode_data_for_ml_training], 
    max_workers=12,  
    flows_outputdir=str(flows_outputdir),
    ml_task="Private HOOPS Embedings Model",
    export_visualization=False
)

# Run the flow to process all files
flow_output, output_dict, flow_file = data_prep.process(inputs={'cad_datasources': [str(datasources_dir)]}, clean_ouput_dir=True)

Cleaning up existing flow directory: C:\Users\LuisSalazar.LY-LS-LEGION\Documents\repos\HOOPS-AI-tutorials\notebooks\out\flows\HOOPS_Embedding_Training
Removing all previous outputs for flow 'HOOPS_Embedding_Training' to avoid build conflicts.


DATA INGESTION:   0%|          | 0/1 [00:00<?, ?it/s]

DATA TRANSFORMATION:   0%|          | 0/10896 [00:00<?, ?it/s]

Total number of items with errors: 291 (2.67%)
Corrupted items are listed in 'C:\Users\LuisSalazar.LY-LS-LEGION\Documents\repos\HOOPS-AI-tutorials\notebooks\out\flows\HOOPS_Embedding_Training\error_summary.json'.
  [218x] Data extraction error: division by zero
  [9x] Data extraction error: "Data key 'graph' does not exist in Zarr storage."
  [26x] Timeout (CUMULATIVE): item exceeded time_limit_s=120.0s (total_elapsed=120.1s). Worker restarted.
  [23x] Timeout (CUMULATIVE): item exceeded time_limit_s=120.0s (total_elapsed=120.2s). Worker restarted.
  [11x] Timeout (CUMULATIVE): item exceeded time_limit_s=120.0s (total_elapsed=120.0s). Worker restarted.
  [3x] Data extraction error: Exception: Failed
  [1x] Data extraction error: list index out of range


[DatasetInfo] Warning: 156 .json files have no matching .data
[DatasetInfo] Skipped 156 files without valid IDs
[DatasetMerger] Using streaming merge into temporary directory store for large dataset...


DATA STORING/LOADING:   0%|          | 0/35640 [00:00<?, ?files/s]

Sequential Task end=====================


## Step 2: Split the dataset

Split the encoded dataset into training, validation, and test sets. The split is
stratified on `face_types` (the distribution of B-rep face categories) so each
partition reflects the full variety of the dataset. Adjust the ratios to match the
size of your dataset — larger datasets can afford smaller validation and test slices.

In [5]:
flow_root_dir = flows_outputdir.joinpath("flows", flow_name)
myFlow_info        = str(flow_root_dir.joinpath(f"{flow_name}.infoset"))
myFlow_dataset     = str(flow_root_dir.joinpath(f"{flow_name}.dataset"))

In [6]:
from hoops_ai.dataset import DatasetLoader

# Load the already encoded dataset and perform the split
datasetLoader = DatasetLoader(merged_store_path = myFlow_dataset, parquet_file_path=myFlow_info)

datasetLoader.split(key='face_types', group="faces",train=0.8, validation=0.1, test=0.1)

[DatasetExplorer] Default local cluster started: <Client: 'tcp://127.0.0.1:52062' processes=1 threads=16, memory=7.45 GiB>


Processing file info:   0%|          | 0/35640 [00:00<?, ?it/s]

DEBUG: Successfully built file lists with 35640 files out of 35640 original file codes

DATASET STRUCTURE OVERVIEW

Group: edges
------------------------------
  edge_convexities: (4293820,) (int32)
  edge_dihedral_angles: (4293820,) (float32)
  edge_indices: (4293820,) (int32)
  edge_lengths: (4293820,) (float32)
  edge_types: (4293820,) (int32)
  edge_u_grids: (4293820, 10, 6) (float32)
  file_id_code_edges: (4293820,) (int64)

Group: faces
------------------------------
  face_areas: (1696502,) (float32)
  face_centroids: (1696502, 3) (float32)
  face_discretization: (1696502, 100, 7) (float32)
  face_indices: (1696502,) (int32)
  face_loops: (1696502,) (int32)
  face_types: (1696502,) (int32)
  file_id_code_faces: (1696502,) (int64)

Group: graph
------------------------------
  edges_destination: (4293820,) (int32)
  edges_source: (4293820,) (int32)
  file_id_code_graph: (4293820,) (int64)
  num_nodes: (4293820,) (int32)

Dataset split by face_types: Train=28529, Validation=3541, 

(28529, 3541, 3570)

## Step 3: Train and save the model

`EmbeddingFlowModel` and `FlowTrainer` are imported from `hoops_ai.ml.EXPERIMENTAL`.
The EXPERIMENTAL namespace signals that the training API may evolve between releases;
the inference API (`HOOPSEmbeddings`, `CADSearch`) is stable.

`FlowTrainer` wraps PyTorch Lightning and handles the training loop, checkpointing,
and logging. The checkpoint written to `trained_model_path` at the end of training
can be registered directly with `HOOPSEmbeddings.register_model()` and used for
similarity search exactly like the pre-trained model bundled in the release.

In [7]:
from hoops_ai.ml.EXPERIMENTAL import EmbeddingFlowModel, FlowTrainer

EmbeddingModel = EmbeddingFlowModel(result_dir= str(pathlib.Path(flows_outputdir).joinpath("flows").joinpath(flow_name)),
                                     log_file= str(pathlib.Path(flows_outputdir).joinpath("flows").joinpath(flow_name).joinpath("flow.log")) )

flow_trainer = FlowTrainer(

    flowmodel       = EmbeddingModel, 
    datasetLoader   = datasetLoader,
    experiment_name = "HOOPS_AI_train",
    result_dir      = flow_root_dir,
    accelerator     = 'gpu',
    devices         = [0], #[0]
    max_epochs      = 10, # 
    batch_size      = 8,
)

trained_model_path = flow_trainer.train()

HOOPS Embedding Model


INFO:GPU available: True (cuda), used: True
INFO:TPU available: False, using: 0 TPU cores


-----------------------------------------------------------------------------------
HOOPS Embedding Model - TRAINING STEP
-----------------------------------------------------------------------------------
Training batch size               : 8
Learning rate                     : (from model default)
Train shuffle                     : True
Train seed                        : None

Train set contains                : 28529 samples (80.05%)
Validation set contains           : 3541 samples (9.94%)
Test set contains                 : 3570 samples (10.02%)
Total samples                     : 35640
Max Epoch                         : 10
Resume checkpoint                 : None

The trained model: C:\Users\LuisSalazar.LY-LS-LEGION\Documents\repos\HOOPS-AI-tutorials\notebooks\out\flows\HOOPS_Embedding_Training\flowtrainer\HOOPS_AI_train\0622\214629\best.ckpt

To monitor the logs, run:
tensorboard --logdir C:\Users\LuisSalazar.LY-LS-LEGION\Documents\repos\HOOPS-AI-tutorials\notebooks\out\flows\

INFO:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Training: |          | 0/? [00:00<?, ?it/s]

...

INFO:Metric eval_loss improved by 0.008 >= min_delta = 0.0. New best score: 0.207


INFO: New best -> best.ckpt (eval_loss=0.21545422)
INFO: Starting epoch 9 with LR: [0.0003]


Validation: |          | 0/? [00:00<?, ?it/s]

INFO:Metric eval_loss improved by 0.005 >= min_delta = 0.0. New best score: 0.203
INFO:`Trainer.fit` stopped: `max_epochs=10` reached.


INFO: New best -> best.ckpt (eval_loss=0.20736365)
INFO: New best -> best.ckpt (eval_loss=0.20265698)


## Next steps

Your checkpoint is ready to use anywhere the pre-trained model is used:

```python
from hoops_ai.ml.embeddings import HOOPSEmbeddings

HOOPSEmbeddings.register_model(
    model_name="My Custom Model",
    checkpoint_path=str(trained_model_path),
)
```

- **Build a similarity index** — see
  [`demo_HOOPS_Embeddings_indexing.ipynb`](./demo_HOOPS_Embeddings_indexing.ipynb)
  to embed your full dataset and write a FAISS index with your custom model.
- **Run CAD similarity search** — see
  [`5b_cad_search_using_HOOPS_embeddings.ipynb`](../notebooks/5b_cad_search_using_HOOPS_embeddings.ipynb)
  to query the index and retrieve similar parts.
- **Predict missing PLM metadata** — see
  [`demo_HOOPS_Embeddings_+_Context_Layer.ipynb`](./demo_HOOPS_Embeddings_+_Context_Layer.ipynb)
  to combine the index with the Context Layer for metadata inference.